# Python Code to Extract the Treasury IR Curve (via FRED)

##### This example:
#####  - Uses FRED_API_KEY environment variable
#####  - Pulls the standard Treasury maturities
#####  - Builds a single yield curve DataFrame

### Step 1: Imports and setup

In [1]:
# import Libs
import os
import requests
import pandas as pd

# import modules
from io import StringIO
from datetime import datetime

### Step 2: Define FRED series (Treasury pillars)

In [2]:
FRED_SERIES = {
    "1M":  "DGS1MO",
    "3M":  "DGS3MO",
    "6M":  "DGS6MO",
    "1Y":  "DGS1",
    "2Y":  "DGS2",
    "3Y":  "DGS3",
    "5Y":  "DGS5",
    "7Y":  "DGS7",
    "10Y": "DGS10",
    "20Y": "DGS20",
    "30Y": "DGS30",
}


### Step 3: Function to download one series

In [3]:
def fetch_fred_series(series_id: str, start: str, end: str) -> pd.Series:
    api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise RuntimeError("FRED_API_KEY not found in environment")

    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",   # use JSON to avoid CSV parsing issues
        "observation_start": start,
        "observation_end": end,
    }

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"{series_id} failed ({r.status_code}): {r.text[:250]}")

    # If FRED returns XML/HTML by mistake, it won't parse as JSON
    try:
        payload = r.json()
    except Exception:
        raise RuntimeError(f"{series_id} returned non-JSON content: {r.text[:250]}")

    if "error_code" in payload or "error_message" in payload:
        raise RuntimeError(f"{series_id} error: {payload}")

    obs = payload.get("observations", [])
    if not obs:
        raise RuntimeError(f"{series_id} returned no observations.")

    s = pd.Series(
        {o["date"]: (None if o["value"] == "." else float(o["value"])) for o in obs},
        name=series_id,
        dtype="float64",
    )
    s.index = pd.to_datetime(s.index)
    return s


### Step 4: Build the yield curve DataFrame

In [4]:
start_date = "2025-12-29"
end_date   = "2025-12-29"

yield_curve_df = pd.DataFrame({
    tenor: fetch_fred_series(series_id, start_date, end_date)
    for tenor, series_id in FRED_SERIES.items()
}).sort_index()

mb_US_T_12292025 = yield_curve_df.T.reset_index()



mb_US_T_12292025.columns = ["US_Treasury_Pillar", "Yield"]
mb_US_T_12292025

,US_Treasury_Pillar,Yield
0,1M,3.69
1,3M,3.68
2,6M,3.59
3,1Y,3.48
4,2Y,3.45
5,3Y,3.51
6,5Y,3.67
7,7Y,3.88
8,10Y,4.12
9,20Y,4.75


### Step 5 - Save the dataframe to a CSV file

In [5]:
# get current date and time
current_datetime = datetime.now()
print("Current date & time : ", current_datetime)
filename1 = datetime.now().strftime("%Y-%m-%d %H-%M-%S-%f")

# create a file object along with extension
file_name = "mb_US_T_12292025 "+filename1+".csv"
#print (file_name)

# converting to csv
mb_US_T_12292025.to_csv(file_name)
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")
print("File saved named:")
print (file_name)
print(current_datetime)    
print("~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~")

Current date & time :  2025-12-30 17:06:29.281681
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~
File saved named:
mb_US_T_12292025 2025-12-30 17-06-29-282207.csv
2025-12-30 17:06:29.281681
~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~ ~~~~
